In [1]:
#import mysql.connector
#from mysql.connector import Error
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:

import warnings

# Suppress the specific Google Generative AI warning
warnings.filterwarnings("ignore")

try:
    import nextmove as nm
    import metrics as mt
    import movelist as ml
    import boardImage as bI
    import htmlPagetemplate as ht
    import nextmove as nm
    import snark as sn
    import re
    import mysql.connector
    from mysql.connector import Error
    from IPython.display import display, HTML
    from datetime import datetime
    

    
except Exception as e:
    # Catches any other unexpected exceptions
    print(f"An unexpected error occurred: {e}")


In [3]:
#this chunk is to define all values and variables that are passed back to the program from the webpage

SourceSpaceColoum = "e"  #from a - h
SourceSpaceRow = "1"  # from 1 - 8
DestinationSpaceColoum = "g" #from a - h
DestinationSpaceRow = "1"  # from 1 - 8


AllMovesString = "1. d2d4 b7b6 2. c2c4 e7e6 3. b1c3 c8b7 4. a2a3 g7g6 5. h2h3 f8g7 6. g1f3 b7f3 7. e2f3 d8f6 8. f1e2 f6d4 9. d1d4 g7d4"
# string built for current came

BotDifficulty = "1000"  #1000-Easy ,  1500 - Medium,  2500 - Hard,  5000 - Grandmaster
MetricDisplay = "1"   #1 - display metrics   0 - do not display
RemarkType = "first move"  #1) first move  2) general move  3)Black won 4) UniqueMovePosition 5)Illegal Move
SnarkLevel = "neutral" #off, neutral, positive, evil


In [4]:
#call NextMove

BoardRow1 = "r,n,b,q,k,b,n,r";    #row 1 a through h
BoardRow2 = "p,p,p,p,p,p,p,p";
BoardRow3 = "0,0,0,0,p,0,0,0";
BoardRow4 = "0,0,0,0,0,0,0,0";
BoardRow5 = "0,0,0,0,0,0,0,0";
BoardRow6 = "0,0,0,0,0,0,0,0";
BoardRow7 = "P,P,P,P,P,P,P,P";
BoardRow8 = "R,N,B,Q,K,B,N,R";   # row 8  a through h

#Boardlayout is 8x8 array (where BoardLayout[x][y] is addressing  (row x+1) (column y+1))
BoardLayout = [BoardRow1.split(","),BoardRow2.split(","),BoardRow3.split(","),BoardRow4.split(","),BoardRow5.split(","),BoardRow6.split(","),BoardRow7.split(","),BoardRow8.split(",")]

UserMove = SourceSpaceColoum +  SourceSpaceRow +  DestinationSpaceColoum + DestinationSpaceRow

#connect to database
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='CHESSBOT' # Optional: leave out to create DB first
   )

if connection.is_connected():
    cursor = connection.cursor()
else:
    exit()

BotMove,Metrics,BoardLayout = nm.nextmove(BoardLayout,AllMovesString,UserMove,cursor,BotDifficulty)
print('here')


1. d2d4 b7b6 2. c2c4 e7e6 3. b1c3 c8b7 4. a2a3 g7g6 5. h2h3 f8g7 6. g1f3 b7f3 7. e2f3 d8f6 8. f1e2 f6d4 9. d1d4 g7d4 10. e1g1
SELECT * FROM moves WHERE  Set5 = '1. d2d4 b7b6 2. c2c4 e7e6 3. b1c3 c8b7 4. a2a3 g7g6 5. h2h3 f8g7' AND Set10 LIKE '6. g1f3 b7f3 7. e2f3 d8f6 8. f1e2 f6d4 9. d1d4 g7d4 10. e1g1%' 
Total Games found: 1
selected BotMove:  d4c3
calling board update
done board update
BotMove:  d4c3
Metrics:  [1, [['d4c3', 1.0]]]
BoardLayout:  [['r', '0', 'b', '0', '0', 'r', 'k', '0'], ['0', 'p', '0', '0', 'b', 'p', 'p', '0'], ['p', '0', 'B', '0', 'p', 'p', '0', 'p'], ['0', '0', 'p', '0', '0', '0', '0', '0'], ['0', '0', '0', '0', '0', '0', '0', '0'], ['0', 'P', '0', '0', 'P', '0', 'P', '0'], ['P', '0', 'P', 'P', '0', 'P', '0', 'P'], ['R', 'N', '0', '0', 'K', '0', 'N', 'R']]
here


In [5]:
#call metrics
BoardLayout = mt.metrics(Metrics, BotMove, BoardLayout)

In [6]:
#obtain webpage base template

FirstPartOfPage = ht.htmlPagetemplate(1)
SecondPartOfPage = ht.htmlPagetemplate(2)
ThirdPartOfPage = ht.htmlPagetemplate(3)
FourthPartOfPage = ht.htmlPagetemplate(4)
FifthPartOfPage = ht.htmlPagetemplate(5)



In [7]:
# append AllMovesString with new pair of moves
move_numbers = re.findall(r'(\d+)\.', AllMovesString)
next_num = int(move_numbers[-1]) + 1 if move_numbers else 1
AllMovesString += f" {next_num}. {UserMove} {BotMove}"
print(AllMovesString)

#call movelist
BaseMovesList = ml.movelist(AllMovesString)

for row in BaseMovesList:
    SecondPartOfPage += '<tr>'
    SecondPartOfPage += '<td style="padding: 6px; border: 2px solid #ddd; font-weight: bold;">'
    SecondPartOfPage += str(row[0])
    SecondPartOfPage += '</td><td style="padding: 6px; border: 2px solid #ddd;">'
    SecondPartOfPage += row[1]
    SecondPartOfPage += '</td><td style="padding: 6px; border: 2px solid #ddd;">'
    SecondPartOfPage += row[2]    
    SecondPartOfPage += '</td>'
    SecondPartOfPage += '</tr>'



1. d2d4 b7b6 2. c2c4 e7e6 3. b1c3 c8b7 4. a2a3 g7g6 5. h2h3 f8g7 6. g1f3 b7f3 7. e2f3 d8f6 8. f1e2 f6d4 9. d1d4 g7d4 10. e1g1 d4c3


In [8]:
#call boardimage ~ 64 times
for row in range(8, 0, -1):  # Corrected: start at 8, stop at 1, step -1
    FirstPartOfPage += '<tr>'
    for column in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
        space_address = f"{column}{row}" 

        # Call your boardImage function
        PiecePictureName = bI.boardImage(BoardLayout, space_address)
        
        # Debug print
#        print(f"{space_address} -> {PiecePictureName}")

        FirstPartOfPage += f'<td><img src="{PiecePictureName}" width="56" alt="Chess Square {space_address}"></td>'
    FirstPartOfPage += '</tr>'


In [9]:
#call snark remark

RemarkText = sn.snark(RemarkType,SnarkLevel,len(BaseMovesList))

FifthPartOfPage = RemarkText + FifthPartOfPage

In [10]:
#output webpage
display(HTML(FirstPartOfPage+SecondPartOfPage+ThirdPartOfPage+FourthPartOfPage+FifthPartOfPage))


In [11]:
#output webpage raw
print(FirstPartOfPage)
print(SecondPartOfPage)
print(ThirdPartOfPage)
print(FourthPartOfPage)
print(FifthPartOfPage)

<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Chess Bot by Rita Group</title></head><body><div style="font-size: 34px; background-color: black; color: white; width: 100%; text-align: center;"><h1><BIG>Welcome to Chess Bot </BIG></h1><table style="font-size: 24px; background-color: black; color: white; margin: 0 auto; width: 50%; border-collapse: collapse; text-align: center;"><tbody><tr><td><table><tbody><tr><td><img src="ChessBoard/Black_Rook_Black.jpg" width="56" alt="Chess Square a8"></td><td><img src="ChessBoard/White_Knight_Black.jpg" width="56" alt="Chess Square b8"></td><td><img src="ChessBoard/Black.jpg" width="56" alt="Chess Square c8"></td><td><img src="ChessBoard/White.jpg" width="56" alt="Chess Square d8"></td><td><img src="ChessBoard/Black_King_Black.jpg" width="56" alt="Chess Square e8"></td><td><img src="ChessBoard/White.jpg" width="56" alt="Chess Square f8"></td><td><img src="ChessBoard/Black_Knight_Black.jpg" width="56" alt="Chess Square g8"></td><td><img src